# Credit Risk — Predicting Loan Default

A worked classification study on the Kaggle
[credit risk dataset](https://www.kaggle.com/datasets/laotse/credit-risk-dataset)
(`laotse/credit-risk-dataset`): 32,581 granted loans described by twelve columns of
applicant, credit-bureau and underwriting information.

The dataset ships without a designated label — it is a bank's operational record — so the
first job is to argue for a target rather than look one up.

## Contents

| § | Section |
| --- | --- |
| 1 | [Setup](#1.-Setup) |
| 2 | [Choosing the dependent variable](#2.-Choosing-the-dependent-variable) |
| 3 | [How the variables relate](#3.-How-the-variables-relate) |
| 4 | [Three classifiers, cross-validated](#4.-Three-classifiers,-cross-validated) |
| 5 | [Hyperparameter optimisation](#5.-Hyperparameter-optimisation) |
| 6 | [Gradient boosting](#6.-Gradient-boosting) |
| 7 | [Robustness and what tuning does not buy](#7.-Robustness-and-what-tuning-does-not-buy) |
| 8 | [Summary](#8.-Summary) |

The pipeline itself lives in `src/`, so this notebook and `run_analysis.py` share one
implementation instead of drifting apart. `src/experiments.py` holds the search spaces;
`src/robustness.py` holds the diagnostics in §7.

---
## 1. Setup

### 1.1 Imports and configuration

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
%matplotlib inline

FIGURES = Path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

### 1.2 Load and clean

In [ ]:
import eda
import robustness as rb
from data import TARGET, UNDERWRITING_FEATURES, load_credit_risk
from experiments import (
    RANDOM_STATE, cross_validate_baselines, gradient_boosting_spec,
    search_specs, search_trace, tune_all,
)
from preprocess import clean, feature_columns

dataset = load_credit_risk()
frame, clean_report = clean(dataset.frame)

print(f"source         : {dataset.source}")
print(f"rows x columns : {frame.shape[0]:,} x {frame.shape[1]}")
print(f"default rate   : {frame[TARGET].mean():.3%}")
print(f"cleaning       : {clean_report}")

Cleaning removes only records that are *impossible* — ages above 100, employment histories
longer than a working life. Extreme-but-possible values (a $6M income, a loan worth 83% of
income) are kept: those are real applicants and the model has to cope with them.

> **Which data is this?** `load_credit_risk()` resolves in three steps: the real CSV in
> `data/`, then a `kagglehub` download, then a calibrated stand-in. If the line above reads
> `synthetic`, Kaggle was unreachable and **these numbers come from the stand-in, not the
> real file** — it reproduces the real marginals and conditional default rates but not the
> sharper joint structure, so absolute scores here run lower than the real data supports.
> Drop the real `data/credit_risk_dataset.csv` in and re-run to regenerate everything.

---
## 2. Choosing the dependent variable

Nothing in the file is marked as a label. A usable target has to be:

1. an **outcome**, not an input to the process that produced the row;
2. **unknown at decision time** — otherwise there is nothing to predict;
3. **low-cardinality** enough to serve as a classification label.

Only one column clears all three.

### 2.1 Column profile

In [ ]:
display(eda.profile(frame))

### 2.2 Candidate targets

Grouping the columns by what they actually are makes the choice obvious: applicant
attributes, application terms and bureau attributes are all *inputs*, and the lender's grade
and interest rate are its own pricing decision. One column is left.

In [ ]:
display(eda.target_candidates(frame))

### 2.3 Class balance

In [ ]:
counts = frame[TARGET].value_counts().sort_index()
print(counts.to_string())
print(f"\nbase rate = {frame[TARGET].mean():.3%}")
print(f"a 'nobody defaults' classifier already scores {1 - frame[TARGET].mean():.2%} accuracy")

path = FIGURES / "01_target_balance.png"
if path.exists():
    display(Image(str(path)))

### 2.4 Conclusion

**The dependent variable is `loan_status`** — 1 = defaulted, 0 = repaid.

Two consequences shape everything downstream:

- **Classes are imbalanced ~78/22**, so accuracy is not a usable score. ROC-AUC is the
  selection metric here, with average precision, F1, balanced accuracy and the Brier score
  reported alongside because they answer different questions.
- **This file holds only loans that were already granted.** There are no declined applicants
  in it, so it supports modelling *default risk on approved loans* — an input to a granting
  decision, not the decision itself. That gap is survivorship bias, and §7.7 returns to it.

---
## 3. How the variables relate

Three questions: how strongly each feature relates to the target, which features duplicate
each other, and — the one that matters most here — which features are *consequences* of the
thing being predicted rather than independent measurements of the applicant.

### 3.1 Strength of association with the target

Point-biserial correlation for numeric columns, Cramér's V for categoricals, and mutual
information on the encoded matrix so the two families are directly comparable.

In [ ]:
assoc = eda.association_with_target(frame)
display(assoc)

### 3.2 Default rate by category

In [ ]:
for col, table in eda.default_rate_by_category(frame).items():
    print(f"\n{col}")
    print(table.to_string(index=False))

### 3.3 Correlation and redundancy

In [ ]:
display(eda.numeric_correlations(frame))
display(eda.redundancy_check(frame))

### 3.4 Distributions

In [ ]:
eda.write_figures(frame, FIGURES)
for name in ["02_default_rate_by_category.png",
             "03_numeric_by_outcome.png",
             "04_spearman_correlation.png"]:
    display(Image(str(FIGURES / name)))

### 3.5 What the structure means

**1. `loan_grade` and `loan_int_rate` are the lender's verdict, not the applicant's profile.**
They are the two strongest predictors precisely *because* an underwriter already compressed
the applicant's risk into them. That makes them **post-treatment variables**: legitimate if
the model runs *after* grading, leakage-like if it is meant to *replace* grading. §4.4 scores
both feature sets so the difference is measured rather than assumed.

**2. `loan_int_rate` is close to a lookup on `loan_grade`.** Harmless for tree ensembles; in
a linear model it splits one effect across two coefficients so neither looks important on its
own.

**3. `loan_percent_income` is `loan_amnt / person_income` by construction** — and it is the
strongest *applicant-side* signal. That is the substantive finding: what predicts default is
neither income nor loan size alone, but the ratio between them. Debt service capacity.

**4. `person_age` and `cb_person_cred_hist_length` are near-duplicates** — a credit file
cannot predate adulthood, so one of the two is largely redundant.

**5. Missingness is not random.** `loan_int_rate` and `person_emp_length` both have gaps.
Imputation is fitted *inside* each CV fold, never on the full dataset, so imputed values
never carry information from the validation split.

**6. Renters default far more than owners, and a prior default on file roughly doubles the
rate.** Collateral and past behaviour dominate — what the credit literature would predict.

---
## 4. Three classifiers, cross-validated

### 4.1 Protocol

- A stratified **20% test set is split off first** and touched exactly once, in §6.4.
- **5-fold stratified cross-validation** on the remaining 80%; stratification holds the ~22%
  default rate in every fold.
- Imputation, scaling and encoding live **inside** the pipeline and are re-fitted per fold.
  Fitting them on the full dataset first would leak validation statistics into training.
- Three learners spanning three inductive biases: linear (**logistic regression**), local
  non-parametric (**k-nearest neighbours**), non-linear ensemble (**random forest**).

### 4.2 Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    frame.drop(columns=[TARGET]), frame[TARGET],
    test_size=0.20, stratify=frame[TARGET], random_state=RANDOM_STATE,
)
full_cols = feature_columns(include_underwriting=True)
app_cols = feature_columns(include_underwriting=False)

print(f"train {len(X_train):,} ({y_train.mean():.2%} default)")
print(f"test  {len(X_test):,} ({y_test.mean():.2%} default)")
print(f"\nall features     ({len(full_cols)}): {full_cols}")
print(f"application-only ({len(app_cols)}): {app_cols}")
print(f"withheld as post-treatment: {UNDERWRITING_FEATURES}")

### 4.3 Baseline results, default settings

In [ ]:
baseline = cross_validate_baselines(X_train, y_train, full_cols)
display(baseline)

### 4.4 Withholding the lender's grade and rate

Predicting from what the applicant actually reports, with the underwriter's verdict removed:

In [ ]:
baseline_app = cross_validate_baselines(X_train, y_train, app_cols)
display(baseline_app)

delta = (baseline.set_index("model")["roc_auc_mean"]
         - baseline_app.set_index("model")["roc_auc_mean"])
print("ROC-AUC lost when loan_grade and loan_int_rate are withheld:")
print(delta.round(4).to_string())

The drop is the share of apparent performance that comes from the underwriter's verdict
rather than from anything the applicant reported. It is smaller than one might expect, which
says the grade is largely *derived from* the same applicant attributes the model already
sees — a summary of the inputs, not new information on top of them.

---
## 5. Hyperparameter optimisation

Three hyperparameters per classifier, each spanning a real range rather than a neighbourhood
of the default. Same folds and same seed as §4, so the comparison is like-for-like.

### 5.1 Search spaces

In [ ]:
specs = search_specs()
for s in specs:
    print(f"{s.name}  ({s.strategy} search{'; ' + s.notes if s.notes else ''})")
    for k, why in s.tuned.items():
        print(f"    {k:20s} {str(s.param_grid['clf__' + k]):38s} {why}")
    print()

### 5.2 Results

In [ ]:
tuned, searches = tune_all(specs, X_train, y_train, full_cols)
display(tuned[["model", "n_candidates", "best_params", "roc_auc_mean", "roc_auc_std",
               "pr_auc_mean", "train_roc_auc_mean", "overfit_gap", "search_seconds"]])

### 5.3 Search traces

In [ ]:
for name, search in searches.items():
    print(f"\n=== {name} — top candidates ===")
    display(search_trace(search))

### 5.4 What tuning bought

In [ ]:
comparison = pd.DataFrame({
    "baseline": baseline.set_index("model")["roc_auc_mean"],
    "tuned": tuned.set_index("model")["roc_auc_mean"],
})
comparison["delta"] = (comparison["tuned"] - comparison["baseline"]).round(4)
display(comparison)

Watch the `overfit_gap` column in §5.2 — train AUC minus cross-validated AUC. A large gap on
a model whose *cross-validated* score still improved is the signature of a learner that
memorises its training set: k-NN with `weights='distance'` reproduces training labels
exactly. The CV score is what counts, but the gap is worth seeing, because it is the same
mechanism that makes a tuned score optimistic — the subject of §7.2.

---
## 6. Gradient boosting

### 6.1 Why this model

`HistGradientBoostingClassifier` — scikit-learn's histogram-based booster, the same family as
LightGBM. It fits this problem well: mixed numeric and categorical features, a strongly
non-linear and interacting relationship between `loan_percent_income` and `loan_grade`, and
insensitivity to the heavy right tail of `person_income` that the linear model needs
rescaled. It runs through the same preprocessing pipeline as the other three so the
comparison stays like-for-like.

The number of boosting rounds is **not** searched: early stopping on an internal validation
split sets it per fit, which is cheaper and less prone to overfitting the search.

### 6.2 Tuning

In [ ]:
gb_spec = gradient_boosting_spec()
print(f"{gb_spec.name}  ({gb_spec.strategy} search over {gb_spec.n_iter} candidates)")
print(f"note: {gb_spec.notes}\n")
for k, why in gb_spec.tuned.items():
    print(f"    {k:22s} {str(gb_spec.param_grid['clf__' + k]):26s} {why}")

gb_tuned, gb_searches = tune_all([gb_spec], X_train, y_train, full_cols)
gb_search = gb_searches[gb_spec.name]
best_model = gb_search.best_estimator_
display(gb_tuned[["model", "n_candidates", "best_params", "roc_auc_mean", "roc_auc_std",
                  "pr_auc_mean", "overfit_gap", "search_seconds"]])

In [ ]:
display(search_trace(gb_search, top=8))

### 6.3 Leaderboard

In [ ]:
leaderboard = (pd.concat([tuned, gb_tuned], ignore_index=True)
                 .sort_values("roc_auc_mean", ascending=False))
display(leaderboard[["model", "roc_auc_mean", "roc_auc_std", "pr_auc_mean",
                     "f1_mean", "balanced_accuracy_mean", "brier_mean"]])

rb.plot_model_comparison(baseline, leaderboard, FIGURES / "05_model_comparison.png")
display(Image(str(FIGURES / "05_model_comparison.png")))

### 6.4 Held-out evaluation

Scored once, on data no fold and no search ever saw. Intervals are 2,000-sample percentile
bootstraps.

In [ ]:
y_proba = best_model.predict_proba(X_test[full_cols])[:, 1]

boot = rb.bootstrap_test_metrics(y_test, y_proba, n_boot=2000)
display(boot)

rb.plot_evaluation(y_test, y_proba, FIGURES / "06_holdout_evaluation.png")
display(Image(str(FIGURES / "06_holdout_evaluation.png")))

### 6.5 Choosing a decision threshold

In [ ]:
display(rb.threshold_table(y_test, y_proba))

Ranking quality is threshold-free; an approve/decline policy is not. This table is where the
real trade-off gets made — how many good borrowers you are willing to turn away per default
avoided. No amount of hyperparameter search makes that choice for you.

---
## 7. Robustness and what tuning does not buy

### 7.1 Sample size and the learning curve

32k rows is a comfortable middle, but the **minority class is the binding constraint**: at a
~22% base rate, the effective sample for learning what default looks like is the ~7k
defaulters, not the 32k rows. Hence stratification everywhere, cross-validation rather than a
single split, and real caution about grades F and G, which hold only a few hundred rows
between them.

The learning curve answers whether more rows would help.

In [ ]:
curve = rb.learning_curve_data(best_model, X_train, y_train, full_cols)
display(curve)

rb.plot_learning_curve(curve, FIGURES / "07_learning_curve.png")
display(Image(str(FIGURES / "07_learning_curve.png")))

slope = curve["cv_mean"].iloc[-1] - curve["cv_mean"].iloc[-2]
print(f"change in CV AUC over the final step: {slope:+.4f}")

Flat. The limit is the **information in these twelve columns**, not the number of rows.
Another 30k identical applications would buy almost nothing; new *columns* — payment history,
existing debt, a bureau DTI — would buy a great deal.

### 7.2 Does hyperparameter optimisation guarantee out-of-sample performance?

**No — and it is systematically optimistic.** Three separate reasons:

1. **The reported best score is a maximum over noisy estimates.** A search over *k* candidates
   returns the best of *k* noisy fold-averages, and part of that "best" is a lucky draw
   against those particular folds. §7.3 measures it.
2. **It optimises the metric you chose, on the distribution you happen to have.** Maximising
   ROC-AUC improves *ranking*. It does not improve calibration, and it knows nothing about the
   cost asymmetry between rejecting a good borrower and approving a bad one (§6.5).
3. **It assumes the future resembles the past.** Every guarantee cross-validation offers is
   conditional on new applicants coming from the same distribution. Credit portfolios break
   that routinely — rates move, the lender's own approval policy shifts, a recession arrives.

### 7.3 Nested cross-validation

An outer loop scoring the *whole procedure*, tuning included, on data the search never
touched. The gap against the inner score is the selection bias.

In [ ]:
nested = rb.nested_cv(gb_spec, X_train, y_train, full_cols, n_iter=10)
print(f"tuned inner-CV score (what a naive report quotes) : {gb_search.best_score_:.4f}")
print(f"nested CV (honest estimate of the procedure)      : {nested['mean']:.4f} +/- {nested['std']:.4f}")
print(f"selection optimism                                : {gb_search.best_score_ - nested['mean']:+.4f}")
print(f"outer-fold scores                                 : {nested['outer_scores']}")

### 7.4 Bootstrap intervals

How precise the estimate is at all. Compare the interval width against the gaps between
models before claiming one is better than another.

In [ ]:
auc_ci = boot.set_index("metric").loc["roc_auc"]
spread = leaderboard["roc_auc_mean"].max() - leaderboard["roc_auc_mean"].min()
print(f"held-out ROC-AUC               : {auc_ci['point_estimate']:.4f}  "
      f"95% CI [{auc_ci['ci_lo_2.5']:.4f}, {auc_ci['ci_hi_97.5']:.4f}]")
print(f"CI width                       : {auc_ci['ci_width']:.4f}")
print(f"spread across all tuned models : {spread:.4f}")
print(f"\n-> models separated by less than {auc_ci['ci_width']:.4f} AUC are not "
      f"distinguishable on this data.")

### 7.5 Seed and partition sensitivity

Re-run cross-validation under different fold partitions. If the score swings with the
partition, the ranking of models is noise.

In [ ]:
seeds = rb.seed_sensitivity(best_model, X_train, y_train, full_cols)
display(seeds)
print(f"spread across seeds: {seeds.attrs['spread']:.4f}")

### 7.6 Slice-level evaluation

An average hides the segments. A model can look strong overall while being close to useless
inside the segment where the decision is actually hard.

In [ ]:
subgroups = rb.subgroup_performance(X_test.reset_index(drop=True), y_test.to_numpy(), y_proba)
display(subgroups)

within = subgroups["roc_auc"].dropna()
print(f"per-slice AUC ranges {within.min():.3f} to {within.max():.3f} "
      f"(overall {auc_ci['point_estimate']:.3f})")

Within a single `loan_grade` the model has far less to work with than the headline number
suggests — because the grade itself carries most of the signal. That is the practical meaning
of the post-treatment problem raised in §3.5.

### 7.7 What this dataset cannot support

Two checks belong in production but need data this file does not carry:

- **Out-of-time validation** — train on older vintages, test on newer. The only honest test of
  drift, and impossible here: there are no origination dates.
- **Reject inference** — the file contains only approved loans, so a model deployed to make
  approval decisions would immediately face a population it never saw. Either score rejected
  applications too, or state plainly that the model describes approved-loan risk only.

And one practice worth naming: monitor **calibration** drift, not just AUC. A ranking can stay
good while the absolute probabilities drift, and the book still gets mispriced.

---
## 8. Summary

In [ ]:
summary = pd.DataFrame([
    ("rows after cleaning", f"{len(frame):,}"),
    ("default base rate", f"{frame[TARGET].mean():.2%}"),
    ("dependent variable", TARGET),
    ("best model", leaderboard.iloc[0]["model"]),
    ("tuned CV ROC-AUC", f"{gb_search.best_score_:.4f}"),
    ("nested CV ROC-AUC", f"{nested['mean']:.4f} +/- {nested['std']:.4f}"),
    ("selection optimism", f"{gb_search.best_score_ - nested['mean']:+.4f}"),
    ("held-out ROC-AUC", f"{auc_ci['point_estimate']:.4f} "
                         f"[{auc_ci['ci_lo_2.5']:.4f}, {auc_ci['ci_hi_97.5']:.4f}]"),
    ("bootstrap CI width", f"{auc_ci['ci_width']:.4f}"),
    ("seed spread", f"{seeds.attrs['spread']:.4f}"),
    ("learning-curve final step", f"{slope:+.4f}"),
    ("data source", dataset.source),
], columns=["quantity", "value"])
display(summary)

### Takeaways

1. **`loan_status` is the dependent variable**; everything else is an input or the lender's
   own pricing decision.
2. **`loan_percent_income` is the strongest applicant-side signal** — the ratio matters, not
   income or loan size alone. `loan_grade` and `loan_int_rate` are stronger still, but they
   are the underwriter's verdict rather than independent measurements.
3. **Gradient boosting wins, but not by a defensible margin.** The bootstrap interval on the
   held-out set is wider than the gaps between the tuned models, so they are statistically
   indistinguishable on this data.
4. **Tuning bought a number, not a better model.** Nested CV shows part of the tuned score is
   selection luck, and the learning curve is flat — the ceiling is the feature set, not the
   sample size or the hyperparameters.